In [ ]:
!pip install -q mne matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 70.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving eeg_prep_hedres to eeg_prep_hedres


In [ ]:
%matplotlib inline

import numpy as np
import mne
import matplotlib.pyplot as plt
from google.colab import files

# ---- 1. Load file ----
path = "/content/eeg_prep_hedres"

data = np.loadtxt(path, skiprows=1)
print("Loaded array shape:", data.shape)   # should be (30503, 33)

times_ms = data[:, 0]
eeg_data = data[:, 1:].T                   # (n_channels, n_times)

channel_names = [
    "FPz", "EOG1", "F3", "Fz", "F4", "EOG2", "FC5", "FC1", "FC2", "FC6",
    "T7", "C3", "C4", "Cz", "T8", "CP5", "CP1", "CP2", "CP6", "P7",
    "P3", "Pz", "P4", "P8", "PO7", "PO3", "POz", "PO4", "PO8", "O1", "Oz", "O2"
]

sfreq = 128.0
ch_types = ["eog" if "EOG" in n else "eeg" for n in channel_names]

info = mne.create_info(channel_names, sfreq, ch_types)
raw = mne.io.RawArray(eeg_data * 1e-6, info)  # µV -> V

montage = mne.channels.make_standard_montage("standard_1020")
raw.set_montage(montage, on_missing="warn")

print(raw)
print("Duration (s):", raw.times[-1])

# ---- 2. STATIC plot (works in Colab) ----
# DO NOT use raw.plot() in Colab

t = raw.times                      # seconds
offset = 80e-6                     # 80 µV spacing in Volts

eeg_picks = mne.pick_types(raw.info, eeg=True)
eeg_names = [raw.ch_names[i] for i in eeg_picks]

fig, ax = plt.subplots(figsize=(12, 8))

for i, (ch_idx, name) in enumerate(zip(eeg_picks, eeg_names)):
    trace = raw.get_data(picks=[ch_idx])[0]
    ax.plot(t, trace + i * offset, color="k", linewidth=0.5)
    ax.text(-0.5, i * offset, name, ha="right", va="center", fontsize=8)

ax.set(xlabel="Time (s)", xlim=(0, 10), yticks=[])
ax.set_title("Preprocessed EEG (first 10 s)")
plt.tight_layout()

# Save THEN download
fig.savefig("eeg_traces.pdf", bbox_inches="tight")
plt.show()

files.download("eeg_traces.pdf")